In [10]:
from collections import Counter, defaultdict
import math
import re
import urllib.request
import numpy as np

np.random.seed(42)

# Map target 23 language codes directly to their actual IndicCorpV2 filenames on Hugging Face
LANG_FILE_MAP = {
    "as": "as.txt",
    "bn": "bn.txt",
    "brx": "bd.txt",  # Bodo (bd.txt)
    "doi": "dg.txt",  # Dogri (dg.txt)
    "gom": "gom.txt",  # Konkani
    "gu": "gu.txt",
    "hi": "hi-1.txt",  # Hindi part 1
    "kn": "kn.txt",
    "ks": "ks.txt",
    "mai": "mai.txt",
    "ml": "ml.txt",
    "mni": "mni.txt",
    "mr": "mr.txt",
    "ne": "ne.txt",
    "or": "or.txt",
    "pa": "pa.txt",
    "sa": "sa.txt",
    "sat": "sat.txt",
    "sd": "sd.txt",
    "te": "te.txt",
    "ta": "ta.txt",
    "ur": "ur.txt",
    "en": "en.txt",
}


def sentence_tokenize(text):
    """Splits text paragraphs into individual clean sentences."""
    sentences = re.split(r"(?<=[।॥.!?])\s+", str(text))
    return [s.strip() for s in sentences if len(s.strip()) > 15]


def fetch_dataset_partial_stream(samples_per_lang=1000):
    print("--- 1. Partial Byte-Streaming Sentences from IndicCorpV2 ---")
    data_by_lang = defaultdict(list)

    for lang_code, filename in LANG_FILE_MAP.items():
        print(f"Streaming {lang_code} ({filename})...")
        url = f"https://huggingface.co/datasets/ai4bharat/IndicCorpV2/resolve/main/data/{filename}"
        collected = []

        try:
            # Request only the first 2MB chunk using HTTP Range headers
            req = urllib.request.Request(
                url,
                headers={
                    "User-Agent": "Mozilla/5.0",
                    "Range": "bytes=0-2097152",  # 2 MB buffer is enough for 1,000+ sentences
                },
            )

            with urllib.request.urlopen(req) as response:
                chunk = response.read().decode("utf-8", errors="ignore")
                lines = chunk.splitlines()

                # Process line by line up to 1000 sentences
                for line in lines[:-1]:  # Drop partial last line
                    line_str = line.strip()
                    if not line_str:
                        continue

                    for sent in sentence_tokenize(line_str):
                        collected.append(sent)
                        if len(collected) >= samples_per_lang:
                            break

                    if len(collected) >= samples_per_lang:
                        break

            data_by_lang[lang_code] = collected[:samples_per_lang]
            print(f"  -> Successfully loaded {len(collected)} sentences.")

        except Exception as e:
            print(f"  -> Error streaming {lang_code}: {e}")

    return data_by_lang


def stratified_split(data_by_lang, train_ratio=0.8, val_ratio=0.1):
    train_x, train_y = [], []
    val_x, val_y = [], []
    test_x, test_y = [], []

    label_map = {lang: idx for idx, lang in enumerate(data_by_lang.keys())}

    for lang, sentences in data_by_lang.items():
        if len(sentences) == 0:
            continue
        np.random.shuffle(sentences)
        n = len(sentences)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)

        label = label_map[lang]

        train_x.extend(sentences[:n_train])
        train_y.extend([label] * n_train)

        val_x.extend(sentences[n_train : n_train + n_val])
        val_y.extend([label] * n_val)

        test_x.extend(sentences[n_train + n_val :])
        test_y.extend([label] * (n - n_train - n_val))

    return (
        (train_x, np.array(train_y, dtype=np.int64)),
        (val_x, np.array(val_y, dtype=np.int64)),
        (test_x, np.array(test_y, dtype=np.int64)),
        label_map,
    )


# =====================================================================
# 2. CUSTOM TF-IDF VECTORIZER FROM SCRATCH
# =====================================================================


class CustomTFIDFVectorizer:

    def __init__(self, max_features=25000):
        self.max_features = max_features
        self.vocabulary_ = {}
        self.idf_ = {}

    def _extract_ngrams(self, text):
        tokens = text.lower().split()
        features = []

        # i. Word Unigrams & Bigrams
        features.extend(tokens)
        for i in range(len(tokens) - 1):
            features.append(f"W_{tokens[i]}_{tokens[i+1]}")

        # ii. Character 2, 3, 4-grams
        for n in [2, 3, 4]:
            for i in range(len(text) - n + 1):
                features.append(f"C_{text[i:i+n]}")

        return features

    def fit(self, raw_documents):
        print("\n--- 2. Building Custom TF-IDF Vocabulary & IDF ---")
        doc_count = len(raw_documents)
        df = defaultdict(int)

        for doc in raw_documents:
            feature_set = set(self._extract_ngrams(doc))
            for feature in feature_set:
                df[feature] += 1

        sorted_features = sorted(df.items(), key=lambda x: x[1], reverse=True)[
            : self.max_features
        ]
        self.vocabulary_ = {
            feature: idx for idx, (feature, _) in enumerate(sorted_features)
        }

        for feature, idx in self.vocabulary_.items():
            self.idf_[idx] = math.log((1 + doc_count) / (1 + df[feature])) + 1.0

        return self

    def transform(self, raw_documents):
        num_docs = len(raw_documents)
        num_features = len(self.vocabulary_)
        X = np.zeros((num_docs, num_features), dtype=np.float32)

        for doc_idx, doc in enumerate(raw_documents):
            ngrams = self._extract_ngrams(doc)
            counts = Counter(
                [
                    self.vocabulary_[ngram]
                    for ngram in ngrams
                    if ngram in self.vocabulary_
                ]
            )

            total_terms = max(len(ngrams), 1.0)

            for feat_idx, count in counts.items():
                tf = count / total_terms
                X[doc_idx, feat_idx] = tf * self.idf_[feat_idx]

            row_norm = np.linalg.norm(X[doc_idx])
            if row_norm > 0:
                X[doc_idx] /= row_norm

        return X


# =====================================================================
# 3. CUSTOM LOGISTIC REGRESSION FROM SCRATCH
# =====================================================================


class CustomLogisticRegression:

    def __init__(self, lr=1.0, epochs=100, batch_size=256):
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.weights = None
        self.bias = None

    def _softmax(self, z):
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def fit(self, X, y, num_classes):
        print("\n--- 3. Training Custom Logistic Regression ---")
        num_samples, num_features = X.shape

        if num_samples == 0:
            raise ValueError("Training dataset is empty.")

        self.weights = np.zeros((num_features, num_classes), dtype=np.float32)
        self.bias = np.zeros((1, num_classes), dtype=np.float32)

        y_encoded = np.zeros((num_samples, num_classes), dtype=np.float32)
        y_encoded[np.arange(num_samples), y.astype(int)] = 1.0

        for epoch in range(self.epochs):
            indices = np.arange(num_samples)
            np.random.shuffle(indices)

            for start in range(0, num_samples, self.batch_size):
                end = start + self.batch_size
                b_idx = indices[start:end]

                X_b, y_b = X[b_idx], y_encoded[b_idx]

                logits = np.dot(X_b, self.weights) + self.bias
                probs = self._softmax(logits)

                error = probs - y_b
                dw = np.dot(X_b.T, error) / len(b_idx)
                db = np.sum(error, axis=0, keepdims=True) / len(b_idx)

                self.weights -= self.lr * dw
                self.bias -= self.lr * db

            if (epoch + 1) % 20 == 0 or epoch == 0:
                logits = np.dot(X, self.weights) + self.bias
                loss = -np.mean(
                    np.sum(
                        y_encoded * np.log(self._softmax(logits) + 1e-15),
                        axis=1,
                    )
                )
                print(
                    f"Epoch {epoch+1:3d}/{self.epochs} - Cross-Entropy Loss: {loss:.4f}"
                )

    def predict(self, X):
        logits = np.dot(X, self.weights) + self.bias
        return np.argmax(self._softmax(logits), axis=1)


# =====================================================================
# 4. CUSTOM MACRO-F1 EVALUATION METRIC
# =====================================================================


def compute_macro_f1(y_true, y_pred, num_classes):
    f1_scores = []
    for c in range(num_classes):
        tp = np.sum((y_true == c) & (y_pred == c))
        fp = np.sum((y_true != c) & (y_pred == c))
        fn = np.sum((y_true == c) & (y_pred != c))

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) > 0
            else 0.0
        )
        f1_scores.append(f1)
    return np.mean(f1_scores)


# =====================================================================
# MAIN EXECUTION PIPELINE
# =====================================================================

if __name__ == "__main__":
    data_by_lang = fetch_dataset_partial_stream(samples_per_lang=1000)
    (train_x, train_y), (val_x, val_y), (test_x, test_y), label_map = (
        stratified_split(data_by_lang)
    )

    num_classes = len(label_map)
    print(
        f"\nDataset Ready -> Train: {len(train_x)}, Val: {len(val_x)}, Test: {len(test_x)}"
    )

    vectorizer = CustomTFIDFVectorizer(max_features=25000)
    vectorizer.fit(train_x)

    X_train = vectorizer.transform(train_x)
    X_val = vectorizer.transform(val_x)
    X_test = vectorizer.transform(test_x)

    clf = CustomLogisticRegression(lr=1.0, epochs=100, batch_size=256)
    clf.fit(X_train, train_y, num_classes)

    print("\n--- 4. Evaluation Results ---")
    val_preds = clf.predict(X_val)
    print(
        f"Validation Macro-F1 Score: {compute_macro_f1(val_y, val_preds, num_classes):.4f}"
    )

    test_preds = clf.predict(X_test)
    print(
        f"Test Macro-F1 Score:       {compute_macro_f1(test_y, test_preds, num_classes):.4f}"
    )

--- 1. Partial Byte-Streaming Sentences from IndicCorpV2 ---
Streaming as (as.txt)...
  -> Successfully loaded 1000 sentences.
Streaming bn (bn.txt)...
  -> Successfully loaded 1000 sentences.
Streaming brx (bd.txt)...
  -> Successfully loaded 1000 sentences.
Streaming doi (dg.txt)...
  -> Successfully loaded 1000 sentences.
Streaming gom (gom.txt)...
  -> Successfully loaded 1000 sentences.
Streaming gu (gu.txt)...
  -> Successfully loaded 1000 sentences.
Streaming hi (hi-1.txt)...
  -> Successfully loaded 1000 sentences.
Streaming kn (kn.txt)...
  -> Successfully loaded 1000 sentences.
Streaming ks (ks.txt)...
  -> Successfully loaded 1000 sentences.
Streaming mai (mai.txt)...
  -> Successfully loaded 1000 sentences.
Streaming ml (ml.txt)...
  -> Successfully loaded 1000 sentences.
Streaming mni (mni.txt)...
  -> Successfully loaded 1000 sentences.
Streaming mr (mr.txt)...
  -> Successfully loaded 1000 sentences.
Streaming ne (ne.txt)...
  -> Successfully loaded 1000 sentences.
Strea